# 07 — Demo curation for the predictability dashboard

Seven processed datasets now share one artifact vocabulary: entropy/predictability
(`entropy.parquet`), sensing density (`seq_q.parquet`), baseline / random-walk / deep
leaderboards, routine cards and curated places. That is a lot of surface area — and the
upcoming Streamlit dashboard must not recompute, or even *touch*, most of it: two datasets
ship >100k users and `seq_hourly.parquet` alone reaches **180M rows (yjmob1)** and
**231M rows (gowalla)**.

So before the dashboard exists, this notebook **curates**: it decides what gets presented
(headline numbers, Π distributions, leaderboards, a handful of featured users) and
materialises a small, self-contained package under `data/processed/dashboard/`:

| file | grain | dashboard purpose |
|---|---|---|
| `dataset_summary.parquet` | 1 row / dataset | the landing-page headline table |
| `pi_distributions.parquet` | 1 row / user | Π & entropy histograms, Π-vs-q scatter |
| `leaderboards.parquet` | 1 row / (dataset, model) | predictors compared on one scale |
| `featured_users.parquet` | 1 row / editorial pick | who to zoom into, and why |
| `users/<dataset>/<user>/…` | per-pick extracts | seq, routine cards, places, POI mix |

Everything here is **read-only upstream**: no `src/` changes, no pipeline stage re-runs —
pure selection and reshaping, so the dashboard becomes a thin rendering layer over files it
can load in milliseconds.

## 1. Setup — root discovery and the dataset registry

GeoLife lives flat under `data/processed/`; the other six datasets in
`data/processed/<key>/`. The registry below mirrors `mobility.config.processed_dir()` and
adds the display metadata the dashboard needs (label, city). `load()` returns `None` for
artifacts a dataset doesn't have — yjmob never went through `routines`/`graphs`, and
yjmob1 has no `deep_results.parquet` yet — so every block below degrades gracefully.

In [1]:
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def _find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "data" / "processed").is_dir():
            return p
    raise RuntimeError("project root with data/processed not found")


ROOT = _find_root()
DATA = ROOT / "data" / "processed"
OUT = DATA / "dashboard"
if OUT.exists():  # keep re-runs idempotent: rebuild the package from scratch
    shutil.rmtree(OUT)
(OUT / "users").mkdir(parents=True)

# Dataset registry — keys/paths mirror `mobility.config.processed_dir()`;
# label/city are display metadata for the dashboard.
DATASETS = [
    {"key": "geolife",    "label": "GeoLife",             "city": "Beijing"},
    {"key": "yjmob1",     "label": "YJMob100K (split 1)", "city": "Tokyo"},
    {"key": "yjmob2",     "label": "YJMob100K (split 2)", "city": "Tokyo"},
    {"key": "fsq_nyc",    "label": "Foursquare NYC",      "city": "New York"},
    {"key": "fsq_tky",    "label": "Foursquare Tokyo",    "city": "Tokyo"},
    {"key": "gowalla",    "label": "Gowalla",             "city": "global"},
    {"key": "brightkite", "label": "Brightkite",          "city": "global"},
]
KEYS = [d["key"] for d in DATASETS]


def pdir(key):
    return DATA if key == "geolife" else DATA / key


def load(key, name, columns=None):
    """Read a processed artifact for a dataset; None when the stage didn't run."""
    p = pdir(key) / name
    if not p.exists():
        return None
    return pd.read_parquet(p, columns=columns)


def wmean(df, col, weight="n_eval"):
    """n_eval-weighted mean of a per-user metric (NaN-safe)."""
    m = df[df[col].notna() & (df[weight] > 0)]
    if m.empty:
        return float("nan")
    return float(np.average(m[col], weights=m[weight]))


print(f"project root: {ROOT}")
print(f"curated pkg : {OUT.relative_to(ROOT)}")
for d in DATASETS:
    print(f"  {d['key']:11s} {d['label']:22s} {d['city']}")

project root: /home/kamil/projects/here/limits_of_predictability_in_human_mobility
curated pkg : data/processed/dashboard
  geolife     GeoLife                Beijing
  yjmob1      YJMob100K (split 1)    Tokyo
  yjmob2      YJMob100K (split 2)    Tokyo
  fsq_nyc     Foursquare NYC         New York
  fsq_tky     Foursquare Tokyo       Tokyo
  gowalla     Gowalla                global
  brightkite  Brightkite             global


## 2. The headline table — one row per dataset

Everything the dashboard's landing page needs lives here: panel size (`n_users`,
`n_cohort`), sensing density (`q_median` = share of *unknown* hourly slots — higher means
sparser), trajectory richness (`n_median` distinct r8 cells), the predictability headline
(`pi_actual_*`, Song-comparable) with the frequency-only `pi_unc_*` bound as context, the
Fano compliance check (`fano_frac`), the best classical predictor (n_eval-weighted top-1
across users), and the editorial artifact counts each dataset contributed (routine cards,
curated places).

In [2]:
SUMMARY_COLS = [
    "dataset", "n_users", "n_cohort", "q_median", "n_median", "s_unc_median",
    "pi_unc_median", "pi_unc_mean", "pi_actual_median", "pi_actual_mean", "fano_frac",
    "best_baseline", "best_baseline_top1", "n_cards", "n_places",
]

rows = []
for d in DATASETS:
    key = d["key"]
    ent = load(key, "entropy.parquet")
    q = load(key, "seq_q.parquet")
    coh = load(key, "cohort.parquet")
    base = load(key, "baseline_hourly_results.parquet")
    rut = load(key, "routines.parquet")
    plc = load(key, "places_summary.parquet")

    # best baseline by n_eval-weighted top-1; alphabetical tie-break keeps it deterministic
    scored = sorted(
        ((pred, wmean(g, "top1")) for pred, g in base.groupby("predictor")),
        key=lambda t: (-t[1], t[0]),
    )
    best_pred, best_top1 = scored[0]

    # Fano compliance: S_actual <= S_unc always holds unless the estimate broke down,
    # which inverts to Pi_actual >= Pi_unc; NaN-Pi users (uncomputable entropy) excluded
    ok = ent[ent["Pi_actual"].notna() & ent["Pi_unc"].notna()]
    rows.append({
        "dataset": key,
        "n_users": len(ent),
        "n_cohort": len(coh) if coh is not None else 0,
        "q_median": float(q["q"].median()),
        "n_median": float(ent["N"].median()),
        "s_unc_median": float(ent["S_unc"].median()),
        "pi_unc_median": float(ent["Pi_unc"].median()),
        "pi_unc_mean": float(ent["Pi_unc"].mean()),
        "pi_actual_median": float(ent["Pi_actual"].median()),
        "pi_actual_mean": float(ent["Pi_actual"].mean()),
        "fano_frac": float((ok["Pi_actual"] >= ok["Pi_unc"]).mean()),
        "best_baseline": best_pred,
        "best_baseline_top1": best_top1,
        "n_cards": len(rut) if rut is not None else 0,
        "n_places": len(plc) if plc is not None else 0,
    })

dataset_summary = pd.DataFrame(rows)[SUMMARY_COLS]
dataset_summary.to_parquet(OUT / "dataset_summary.parquet", index=False)
display(dataset_summary.round(3))

,dataset,n_users,n_cohort,q_median,n_median,s_unc_median,pi_unc_median,pi_unc_mean,pi_actual_median,pi_actual_mean,fano_frac,best_baseline,best_baseline_top1,n_cards,n_places
0,geolife,182,76,0.905,28.0,4.009,0.364,0.371,0.553,0.548,0.934,markov2_col,0.228,244,640
1,yjmob1,100000,99974,0.620,132.0,4.891,0.442,0.443,0.639,0.633,1.000,markov1,0.287,0,0
2,yjmob2,25000,24997,0.596,150.0,5.153,0.418,0.421,0.625,0.620,1.000,markov1,0.226,0,0
3,fsq_nyc,1083,62,0.978,32.0,3.776,0.438,0.439,0.573,0.576,0.992,time,0.286,11,706
4,fsq_tky,2293,494,0.978,33.0,3.789,0.439,0.449,0.582,0.593,0.996,freq,0.273,77,2101
5,gowalla,107069,1954,0.980,11.0,3.021,0.333,0.355,0.437,0.435,0.860,time,0.176,72,22040
6,brightkite,8173,2757,0.978,41.0,3.835,0.462,0.487,0.654,0.677,0.976,markov1_col,0.511,914,17409


### Which Π is the headline? The trio, and why

Each user gets three predictabilities, and they are ordered. Entropies are non-increasing —
$S^{rand} \geq S^{unc} \geq S^{actual}$: shuffling a sequence can't create order, and
ignoring temporal order can't reveal it — and the Fano inversion $\Pi(S)$ is *decreasing*
in entropy, so the ceilings are non-decreasing:

$$\Pi_{rand} \leq \Pi_{unc} \leq \Pi_{actual}$$

- **$\Pi_{actual}$ is the headline** because Song et al. (2010)'s famous 0.93 is exactly
  this inversion applied to their phone data's *actual* entropy — the apples-to-apples
  number the whole project re-examines.
- **$\Pi_{unc}$** (frequency-only visits, no order) is the lower-bound *context* column;
  **$\Pi_{rand}$** (uniform chaos) sits near zero and mostly serves to define the scale.
- **`fano_frac`** is therefore a *compliance* check: the share of users where the ordering
  holds as theory demands, $\Pi_{actual} \geq \Pi_{unc}$ (computed where both Π are
  defined; ties count as compliant).

### What the headline says

- **Π ≈ 0.55 on GeoLife (median 0.553, mean 0.548; 0.637 in its curated 76-user cohort)
  and ≈ 0.63 on YJMob100K (mean 0.633 on split 1) — vs Song's 0.93.** Across all seven
  datasets the Π_actual median spans 0.437 (Gowalla) to 0.654 (Brightkite), under four
  sensing technologies. The paper's 0.93 came from undetected exploration; under honest
  missingness the ceiling is ~2/3 everywhere.
- **The `pi_unc_*` bound stays in a narrow 0.33–0.49 band.** What separates datasets is
  how much of that frequency-only floor *temporal order* buys back — the gap between
  Π_unc and Π_actual is the "order dividend" the dashboard can visualise.
- **Density ≠ predictability.** YJMob100K is by far the best-sensed panel (q ≈ 0.60–0.62
  vs 0.90–0.98 elsewhere — i.e. 40% of hourly slots observed vs 2–10%) yet sits mid-pack
  on Π. Sensing density buys coverage, not regularity; this is the project's core finding.
- **`fano_frac` — the Fano compliance check (Π_actual ≥ Π_unc)** — is exactly 1.000 on
  both yjmob grids, ≥ 0.976 on the check-in panels and 0.934 on GeoLife: the bound is
  respected by essentially everyone. Gowalla's 0.860 is its never-revisiting users, where
  the Fano inversion runs at its numerical floor — the dashboard shows the fraction with a
  small-N caveat, not hidden.
- **Editorial artifacts are unevenly available** — routine cards and curated places exist
  only where `routines`/`graphs` ran (everything except yjmob), which is exactly what the
  per-user extracts in §6 have to respect.

## 3. Π distributions — the user-level panel

Histograms, box plots and Π-vs-q scatters need per-user values, not per-dataset summaries.
We concatenate all seven entropy tables and left-join the per-user density `q` from
`seq_q.parquet`. The result (~244k rows) is small enough for the dashboard to load whole —
unlike the 180M-row panels it was derived from.

In [3]:
frames = []
for d in DATASETS:
    key = d["key"]
    ent = load(key, "entropy.parquet")
    q = load(key, "seq_q.parquet")[["user", "q"]]
    frames.append(ent.merge(q, on="user", how="left").assign(dataset=key))

PI_COLS = ["dataset", "user", "N", "S_rand", "S_unc", "S_actual",
           "Pi_rand", "Pi_unc", "Pi_actual", "q"]
pi_all = pd.concat(frames, ignore_index=True)[PI_COLS]
pi_all.to_parquet(OUT / "pi_distributions.parquet", index=False)

print(f"pi_distributions: {len(pi_all):,} rows across {pi_all['dataset'].nunique()} datasets")
display(pi_all.groupby("dataset")[["Pi_unc", "Pi_actual", "q", "N"]].median().round(3))
display(pi_all.head(8))

pi_distributions: 243,800 rows across 7 datasets


,Pi_unc,Pi_actual,q,N
dataset,,,,
brightkite,0.462,0.654,0.978,41.0
fsq_nyc,0.438,0.573,0.978,32.0
fsq_tky,0.439,0.582,0.978,33.0
geolife,0.364,0.553,0.905,28.0
gowalla,0.333,0.437,0.980,11.0
yjmob1,0.442,0.639,0.620,132.0
yjmob2,0.418,0.625,0.596,150.0


,dataset,user,N,S_rand,S_unc,S_actual,Pi_rand,Pi_unc,Pi_actual,q
0,geolife,000,108,6.754888,4.231734,2.748872,0.009259,0.520440,0.719283,0.873653
1,geolife,001,46,5.523562,3.725464,2.739232,0.021739,0.503719,0.668157,0.782334
2,geolife,002,94,6.554589,3.915713,2.233974,0.010638,0.552879,0.775784,0.819572
3,geolife,003,231,7.851749,4.922749,3.005433,0.004329,0.499999,0.725068,0.706595
4,geolife,004,193,7.592457,4.474096,2.868315,0.005181,0.541325,0.732336,0.703709
5,geolife,005,40,5.321928,3.413803,2.404594,0.025000,0.542328,0.709528,0.900456
6,geolife,006,40,5.321928,4.101375,2.992778,0.025000,0.408636,0.615603,0.905852
7,geolife,007,64,6.000000,4.077410,2.734557,0.015625,0.485041,0.691622,0.761866


## 4. Leaderboards — every predictor on every dataset, one scale

Baselines, random walks and deep models all publish per-user metrics in the shared
`n_eval, top1, top3, dist_km` scheme, so they can be stacked into one table: per model,
the n_eval-weighted mean top-1 / top-3 and mean distance error. `source` keeps the family
tag (`baseline` / `rw` / `deep`) so the dashboard can filter; yjmob appears with baselines
only (no graphs stage yet), and yjmob1 additionally lacks a deep run.

In [4]:
LB_COLS = ["dataset", "model", "source", "n_users", "n_eval_sum", "top1", "top3", "dist_km"]
SOURCES = [
    ("baseline_hourly_results.parquet", "predictor", "baseline", {}),
    ("rw_results.parquet", "method", "rw",
     {"top1_acc": "top1", "top3_acc": "top3", "dist_err_km": "dist_km"}),
    ("deep_results.parquet", "model", "deep", {}),
]

rows = []
for d in DATASETS:
    key = d["key"]
    for name, model_col, src, ren in SOURCES:
        df = load(key, name)
        if df is None:
            continue
        df = df.rename(columns=ren)
        for mval, g in df.groupby(model_col, sort=True):
            rows.append({
                "dataset": key, "model": mval, "source": src,
                "n_users": int(g["user"].nunique()),
                "n_eval_sum": int(g["n_eval"].sum()),
                "top1": wmean(g, "top1"), "top3": wmean(g, "top3"),
                "dist_km": wmean(g, "dist_km"),
            })

leaderboards = pd.DataFrame(rows)[LB_COLS]
leaderboards.to_parquet(OUT / "leaderboards.parquet", index=False)
print(f"leaderboards: {len(leaderboards)} model rows across {leaderboards['dataset'].nunique()} datasets")
best = (leaderboards.sort_values("top1", ascending=False)
        .groupby(["dataset", "source"], sort=False).head(1)
        .sort_values(["dataset", "source"]).reset_index(drop=True))
display(best.round(3))
display(leaderboards.round(3))

leaderboards: 81 model rows across 7 datasets


,dataset,model,source,n_users,n_eval_sum,top1,top3,dist_km
0,brightkite,markov1_col,baseline,8173,315710,0.511,0.592,165.468
1,brightkite,temporal_transformer,deep,2729,287102,0.298,0.398,1545.771
2,brightkite,rw_trip_dest,rw,2757,18230,0.262,0.365,430.726
3,fsq_nyc,time,baseline,1083,30102,0.286,0.487,4.593
4,fsq_nyc,temporal_transformer,deep,62,3648,0.224,0.373,5.749
5,fsq_nyc,rw_trip_dest,rw,62,197,0.299,0.416,4.998
6,fsq_tky,freq,baseline,2293,79524,0.273,0.487,5.783
7,fsq_tky,temporal_transformer,deep,494,24445,0.254,0.456,5.617
8,fsq_tky,rw_trip_dest,rw,494,1761,0.261,0.469,6.270
9,geolife,markov2_col,baseline,180,14720,0.228,0.368,295.243


,dataset,model,source,n_users,n_eval_sum,top1,top3,dist_km
0,geolife,freq,baseline,180,14720,0.156,0.288,565.009
1,geolife,markov1,baseline,180,14720,0.185,0.319,395.198
2,geolife,markov1_col,baseline,180,14720,0.227,0.378,295.767
3,geolife,markov2,baseline,180,14720,0.169,0.300,395.395
4,geolife,markov2_col,baseline,180,14720,0.228,0.368,295.243
...,...,...,...,...,...,...,...,...
76,brightkite,time,baseline,8173,315710,0.368,0.519,401.035
77,brightkite,rw_hourly_seq,rw,2729,302363,0.204,0.298,462.233
78,brightkite,rw_trip_dest,rw,2757,18230,0.262,0.365,430.726
79,brightkite,deepmove_rnn,deep,2729,287102,0.278,0.359,1850.045


### The winner flips — and that is the story

- **GeoLife (GPS, Beijing):** `temporal_transformer` (0.235) edges out `markov2_col`
  (0.228) — deep learning only earns its complexity on the richest traces.
- **YJMob100K (grid, Tokyo):** plain `markov1` wins both splits (0.287 / 0.226) and the
  transformer trails badly on yjmob2 (0.135). With dense hourly grids the first-order chain
  is hard to beat; the missing deep run on yjmob1 stays on the backlog.
- **Check-ins (fsq_nyc):** the venue-graph random walk `rw_trip_dest` (0.299) beats every
  classical baseline (best: `time` 0.286) — graph structure carries signal where timestamps
  are sparse.
- **Brightkite:** collective Markov (`markov1_col`, 0.511) nearly doubles the best deep
  model (0.298) — pooling across users dominates when individual histories are short.

The crown changes hands across datasets *and* across model families. No single champion
survives the sweep — which is precisely why the dashboard needs the full leaderboard per
dataset rather than a global "best model" badge.

## 5. Featured users — deterministic editorial picks

The dashboard needs zoom-in stories that are *reproducible*, not hand-picked. Selection
rules (deterministic; ties broken by ascending user id):

1. **`most_predictable` / `most_exploratory`** — per dataset, among users with `N >=` the
   dataset's median `N` (so picks aren't sparse-user artifacts) and a computable
   `Pi_actual`: the highest- and lowest-Π users.
2. **`suggest_flagship`** — GeoLife user `022`, the routing/suggestion demo user — always
   present in the package. This run, rule 1 already picked it (022 happens to be GeoLife's
   most predictable user, Π_actual 0.864), so its rule-1 row stands in and no duplicate is
   added.
3. **`routine_creature`** — the Brightkite user with the most routine cards (if already
   picked by rule 1, the next-most-carded user is taken so every pick stays unique).
   This one may carry an undefined Π — a single-cell user whose entropy isn't computable;
   the cards panel is the story there.

Each pick carries its context (Π_unc / Π_actual / N / q, plus card and place counts) so
the dashboard can explain *why* a user is featured.

In [5]:
RUTS = {d["key"]: load(d["key"], "routines.parquet") for d in DATASETS}
PLCS = {d["key"]: load(d["key"], "places_summary.parquet") for d in DATASETS}

picks = []
for d in DATASETS:
    key = d["key"]
    ent = load(key, "entropy.parquet")
    cand = ent[ent["N"] >= ent["N"].median()].dropna(subset=["Pi_actual"])
    hi = cand.sort_values(["Pi_actual", "user"], ascending=[False, True]).iloc[0]
    picks.append((key, str(hi["user"]), "most_predictable"))
    lo = (cand[cand["user"] != hi["user"]]
          .sort_values(["Pi_actual", "user"], ascending=[True, True]).iloc[0])
    picks.append((key, str(lo["user"]), "most_exploratory"))

if not any(k == "geolife" and u == "022" for k, u, _ in picks):
    picks.append(("geolife", "022", "suggest_flagship"))

taken = {u for k, u, _ in picks if k == "brightkite"}
card_counts = RUTS["brightkite"].groupby("user").size().sort_values(ascending=False)
creature = next(u for u in card_counts.index if u not in taken)
picks.append(("brightkite", str(creature), "routine_creature"))

FEATURED_COLS = ["dataset", "user", "why", "Pi_unc", "Pi_actual", "N", "q", "n_cards", "n_places"]
rows = []
for key, user, why in picks:
    e = pi_all[(pi_all["dataset"] == key) & (pi_all["user"] == user)].iloc[0]
    rut, plc = RUTS[key], PLCS[key]
    rows.append({
        "dataset": key, "user": user, "why": why,
        "Pi_unc": float(e["Pi_unc"]), "Pi_actual": float(e["Pi_actual"]),
        "N": int(e["N"]),
        "q": float(e["q"]) if pd.notna(e["q"]) else float("nan"),
        "n_cards": int((rut["user"] == user).sum()) if rut is not None else 0,
        "n_places": int((plc["user"] == user).sum()) if plc is not None else 0,
    })

featured_users = pd.DataFrame(rows)[FEATURED_COLS]
featured_users.to_parquet(OUT / "featured_users.parquet", index=False)
display(featured_users.round(3))

,dataset,user,why,Pi_unc,Pi_actual,N,q,n_cards,n_places
0,geolife,022,most_predictable,0.543,0.864,104,0.771,1,13
1,geolife,111,most_exploratory,0.065,0.225,194,0.988,1,1
2,yjmob1,66750,most_predictable,0.728,0.951,152,0.394,0,0
3,yjmob1,29085,most_exploratory,0.044,0.202,316,0.791,0,0
4,yjmob2,8872,most_predictable,0.511,0.924,165,0.268,0,0
5,yjmob2,4057,most_exploratory,0.053,0.242,378,0.729,0,0
6,fsq_nyc,185,most_predictable,0.812,0.855,63,0.778,0,2
7,fsq_nyc,882,most_exploratory,0.106,0.274,50,0.991,0,1
8,fsq_tky,2132,most_predictable,0.485,0.908,42,0.980,0,1
9,fsq_tky,2182,most_exploratory,0.122,0.272,45,0.992,0,1


## 6. Per-user extracts — so the dashboard never scans the big panels

Each pick gets its own directory under `users/<dataset>/<user>/` with:

- `seq.parquet` — the hourly sequence enriched with `weekday`, `hour_of_day`, `known`
  (`cell_idx != -1`) and r8 cell centroids (`lat`/`lon` joined from `cells_r8` on `h3`);
- `cards.parquet` / `places.parquet` — that user's routine cards and curated places
  (empty-but-typed frames where the upstream stage didn't run, e.g. yjmob);
- `poi.parquet` — **yjmob only**: POI-category counts summed over the user's distinct known
  r8 cells (the "what kind of places does this person visit" panel).

The big `seq_hourly` panels (yjmob1: **180M rows**, gowalla: **231M rows**) are read with a
pyarrow predicate filter — `filters=[("user", "==", user)]` — so row-group statistics let
arrow skip everything unrelated. Timings are printed per pick; a full scan would take
minutes, the filtered reads take fractions of a second.

In [6]:
RUT_TEMPLATE = load("geolife", "routines.parquet").iloc[:0]        # typed empty frames
PLC_TEMPLATE = load("geolife", "places_summary.parquet").iloc[:0]  # for missing stages
SEQ_COLS = ["user", "hour", "cell_idx", "h3", "weekday", "hour_of_day", "lat", "lon", "known"]
POI_KEYS = {"yjmob1", "yjmob2"}


def extract_user(key, user):
    """Per-user extract: (seq, cards, places, poi|None, filtered_read_seconds)."""
    t0 = time.perf_counter()
    seq = pq.read_table(pdir(key) / "seq_hourly.parquet",
                        filters=[("user", "==", user)]).to_pandas()
    read_s = time.perf_counter() - t0

    seq = seq.merge(load(key, "cells_r8.parquet")[["h3", "lat", "lon"]], on="h3", how="left")
    seq = seq.assign(
        weekday=seq["hour"].dt.weekday.astype("int64"),
        hour_of_day=seq["hour"].dt.hour.astype("int64"),
        known=seq["cell_idx"].ne(-1),
    )[SEQ_COLS].sort_values("hour").reset_index(drop=True)

    rut, plc = RUTS[key], PLCS[key]
    cards = rut[rut["user"] == user] if rut is not None else RUT_TEMPLATE.copy()
    places = plc[plc["user"] == user] if plc is not None else PLC_TEMPLATE.copy()

    poi = None
    if key in POI_KEYS:
        p = load(key, "poi.parquet")
        hit = p[p["h3"].isin(seq.loc[seq["known"], "h3"].unique())]
        if len(hit):  # summed POI mix over the user's distinct known cells, one row
            poi = hit.drop(columns="h3").sum().astype("int64").to_frame().T
    return seq, cards, places, poi, read_s


n_user_files = 0
for r in featured_users.itertuples(index=False):
    seq, cards, places, poi, read_s = extract_user(r.dataset, r.user)
    udir = OUT / "users" / r.dataset / r.user
    udir.mkdir(parents=True, exist_ok=True)
    seq.to_parquet(udir / "seq.parquet", index=False)
    cards.to_parquet(udir / "cards.parquet", index=False)
    places.to_parquet(udir / "places.parquet", index=False)
    if poi is not None:
        poi.to_parquet(udir / "poi.parquet", index=False)
    n_user_files += 3 + (poi is not None)
    tag = "  <- predicate-pushdown read" if r.dataset in ("yjmob1", "gowalla", "brightkite") else ""
    print(f"{r.dataset:11s}/{r.user:7s} ({r.why:16s}) seq {len(seq):6,} rows | "
          f"cards {len(cards):4d} | places {len(places):5d} | filtered read {read_s:5.2f}s{tag}")

y1 = max(t for t in featured_users.itertuples(index=False) if t.dataset == "yjmob1")
_, _, _, _, y1_s = extract_user("yjmob1", y1.user)
print(f"\nslowest yjmob1 filtered read: {y1_s:.2f}s against a 180,000,000-row panel")

geolife    /022     (most_predictable) seq  4,772 rows | cards    1 | places    13 | filtered read  0.02s
geolife    /111     (most_exploratory) seq 20,132 rows | cards    1 | places     1 | filtered read  0.02s
yjmob1     /66750   (most_predictable) seq  1,800 rows | cards    0 | places     0 | filtered read  0.09s  <- predicate-pushdown read


yjmob1     /29085   (most_exploratory) seq  1,800 rows | cards    0 | places     0 | filtered read  0.08s  <- predicate-pushdown read
yjmob2     /8872    (most_predictable) seq  1,800 rows | cards    0 | places     0 | filtered read  0.06s
yjmob2     /4057    (most_exploratory) seq  1,800 rows | cards    0 | places     0 | filtered read  0.06s


fsq_nyc    /185     (most_predictable) seq  7,637 rows | cards    0 | places     2 | filtered read  0.02s
fsq_nyc    /882     (most_exploratory) seq  7,444 rows | cards    0 | places     1 | filtered read  0.03s
fsq_tky    /2132    (most_predictable) seq  6,424 rows | cards    0 | places     1 | filtered read  0.02s
fsq_tky    /2182    (most_exploratory) seq  7,491 rows | cards    0 | places     1 | filtered read  0.02s


gowalla    /19917   (most_predictable) seq  3,275 rows | cards    0 | places     1 | filtered read  0.03s  <- predicate-pushdown read
gowalla    /77759   (most_exploratory) seq  4,361 rows | cards    0 | places     0 | filtered read  0.03s  <- predicate-pushdown read


brightkite /206     (most_predictable) seq 12,866 rows | cards    0 | places     2 | filtered read  0.02s  <- predicate-pushdown read
brightkite /22029   (most_exploratory) seq  3,522 rows | cards    0 | places     1 | filtered read  0.02s  <- predicate-pushdown read
brightkite /9957    (routine_creature) seq  1,133 rows | cards   39 | places     1 | filtered read  0.02s  <- predicate-pushdown read



slowest yjmob1 filtered read: 0.08s against a 180,000,000-row panel


## 7. Verification — reload the whole package from disk

The contract the dashboard can rely on: exact schemas and grains for the four curated
tables, one directory per pick with the expected per-user files, and a sub-second
end-to-end load of the worst-case featured user (yjmob1).

In [7]:
# --- dataset_summary ---------------------------------------------------------
ds = pd.read_parquet(OUT / "dataset_summary.parquet")
assert list(ds.columns) == SUMMARY_COLS
assert len(ds) == 7 and set(ds["dataset"]) == set(KEYS)
assert ds["n_users"].gt(0).all() and ds["best_baseline"].notna().all()
assert ds["best_baseline_top1"].between(0, 1).all()
assert ds["fano_frac"].between(0, 1).all()
assert ds["pi_actual_median"].ge(ds["pi_unc_median"]).all()  # headline ceiling above its floor

# --- pi_distributions --------------------------------------------------------
pi_chk = pd.read_parquet(OUT / "pi_distributions.parquet")
assert list(pi_chk.columns) == PI_COLS
expected_rows = sum(len(load(k, "entropy.parquet")) for k in KEYS)
assert len(pi_chk) == expected_rows, (len(pi_chk), expected_rows)
qq = pd.concat([load(k, "seq_q.parquet").assign(dataset=k) for k in KEYS], ignore_index=True)
chk = pi_chk.merge(qq[["dataset", "user", "q"]], on=["dataset", "user"], how="outer",
                   suffixes=("", "_src"), indicator=True)
assert chk["_merge"].eq("both").all()  # every (dataset, user) pair survives the join
assert chk["q"].notna().eq(chk["q_src"].notna()).all()  # q is NA exactly when upstream says so

# --- leaderboards ------------------------------------------------------------
lb = pd.read_parquet(OUT / "leaderboards.parquet")
assert list(lb.columns) == LB_COLS
assert set(lb["dataset"]) == set(KEYS)
assert set(lb.loc[lb["source"] == "rw", "dataset"]) == {
    k for k in KEYS if (pdir(k) / "rw_results.parquet").exists()}
assert set(lb.loc[lb["source"] == "deep", "dataset"]) == {
    k for k in KEYS if (pdir(k) / "deep_results.parquet").exists()}
assert lb.loc[lb["source"] == "baseline"].groupby("dataset")["model"].nunique().eq(8).all()
assert lb["n_eval_sum"].gt(0).all() and lb[["top1", "top3"]].notna().all().all()

# --- featured users + per-user tree ------------------------------------------
fu = pd.read_parquet(OUT / "featured_users.parquet")
FEATURED_COLS = ["dataset", "user", "why", "Pi_unc", "Pi_actual", "N", "q", "n_cards", "n_places"]
assert list(fu.columns) == FEATURED_COLS
assert not fu.duplicated(["dataset", "user"]).any()
assert fu.groupby("dataset").size().ge(2).all()
assert (fu["why"] == "suggest_flagship").sum() <= 1  # skipped when rule 1 already picked 022
assert fu.loc[(fu["dataset"] == "geolife") & (fu["user"] == "022")].shape[0] == 1
assert (fu["why"] == "routine_creature").sum() == 1
for r in fu.itertuples(index=False):
    udir = OUT / "users" / r.dataset / r.user
    for name in ("seq", "cards", "places"):
        assert (udir / f"{name}.parquet").exists(), udir
    seq_u = pd.read_parquet(udir / "seq.parquet")
    assert list(seq_u.columns) == SEQ_COLS
    assert seq_u["user"].eq(r.user).all() and seq_u["known"].dtype == bool
    assert seq_u["lat"].notna().any(), f"cells_r8 join produced no centroids: {udir}"
    if r.dataset in POI_KEYS:
        assert (udir / "poi.parquet").exists(), udir

# --- worst-case featured-user load, end to end --------------------------------
row = fu.loc[fu["dataset"] == "yjmob1"].iloc[0]
t0 = time.perf_counter()
udir = OUT / "users" / "yjmob1" / row["user"]
seq_u = pd.read_parquet(udir / "seq.parquet")
cards_u = pd.read_parquet(udir / "cards.parquet")
places_u = pd.read_parquet(udir / "places.parquet")
poi_u = pd.read_parquet(udir / "poi.parquet")
load_s = time.perf_counter() - t0
assert load_s < 1.0, f"featured-user load too slow: {load_s:.2f}s"
print(f"featured yjmob1/{row['user']} end-to-end: {load_s * 1000:.0f} ms "
      f"(seq {len(seq_u)} rows, cards {len(cards_u)}, places {len(places_u)}, "
      f"poi {poi_u.shape[0]}x{poi_u.shape[1]})")
print(f"package: 4 curated tables + {n_user_files} per-user parquets "
      f"under {OUT.relative_to(ROOT)}/")
print("CURATED PACKAGE OK")

featured yjmob1/66750 end-to-end: 8 ms (seq 1800 rows, cards 0, places 0, poi 1x85)
package: 4 curated tables + 49 per-user parquets under data/processed/dashboard/
CURATED PACKAGE OK
